# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pangakeerthi11/keerthi-flyrank-ml-internship-2026/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


## 1. Ranked actions + reason codes

The action playbook ranks content pages using the final refresh score, model probability, confidence level, and reason codes.

The ranking is intended to help a content team decide which pages should be reviewed first.

### Reason codes

- stale_visible_page: The page has not been updated recently but still receives impressions.
- declining_with_demand: The page has a declining trend while still receiving search impressions.
- thin_visible_page: The page has relatively little content but receives impressions.
- low_ctr_visible_page: The page receives impressions but has a low click-through rate.
- low_engagement_visible_page: The page receives sessions but shows weak engagement.
- model_decline_risk: The model assigns a higher probability to the declining label.
- visible_model_opportunity: The model identifies a possible review opportunity with sufficient impressions.

### Archetype-to-action mapping

- Thin visible content → Expand and refresh.
- High visibility with low CTR → Refresh and review the title and description.
- Low engagement content → Review content relevance and engagement.
- Aging or declining visible content → Review for a possible refresh.
- No strong intervention signal → Monitor.

### Decay and refresh insight

Older content is not automatically unsuccessful. Age and time since the last update are useful review signals, especially when combined with declining trends, search visibility, CTR, or engagement signals.

These patterns are observed associations, not proof that refreshing a page will improve its performance.

The highest-ranked pages should be reviewed first, but the final action must be decided by a human.


In [8]:

# Check whether the official ranked queue already exists
queue_path = repo_path / "outputs" / "refresh_queue.csv"

if queue_path.exists():
    print("Official ranked queue already exists.")
    print("Skipping pipeline rerun and optional PDF generation.")

else:
    # Run the official pipeline if the queue is missing
    result = subprocess.run(
        [sys.executable, str(repo_path / "scripts" / "run_all.py")],
        cwd=str(repo_path),
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)

        if not queue_path.exists():
            raise RuntimeError(
                "The ranked queue was not generated. "
                "Please check the pipeline output."
            )

# Load the generated ranked queue
queue = pd.read_csv(queue_path)

print("\nRanked queue loaded successfully!")
print("Total ranked pages:", len(queue))

print("\nAvailable columns:")
print(queue.columns.tolist())


Official ranked queue already exists.
Skipping pipeline rerun and optional PDF generation.

Ranked queue loaded successfully!
Total ranked pages: 30000

Available columns:
['final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_name', 'best_model_probability', 'baseline_refresh_score', 'confidence', 'suggested_action', 'final_reason_codes', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


In [9]:

import pandas as pd

# 1. Identify the reason-code column automatically
reason_columns = [
    col for col in queue.columns
    if "reason" in col.lower()
]

print("Reason-code columns found:", reason_columns)

if not reason_columns:
    raise ValueError(
        "No reason-code column found. Check the available queue columns."
    )

reason_col = reason_columns[0]

# 2. Create content archetypes from actual reason codes
def classify_archetype(reason_text):

    reasons = str(reason_text).lower()

    if "thin" in reasons:
        return "thin_visible_content"

    elif "ctr" in reasons:
        return "high_visibility_low_ctr"

    elif "engagement" in reasons:
        return "low_engagement_content"

    elif any(term in reasons for term in [
        "stale", "declining", "decay", "age"
    ]):
        return "aging_or_declining_visible"

    else:
        return "general_monitoring"


queue["content_archetype"] = queue[
    reason_col
].apply(classify_archetype)

# 3. Map archetypes to recommended actions
archetype_action_map = {
    "thin_visible_content": "expand_and_refresh",
    "high_visibility_low_ctr": "review_title_and_description",
    "low_engagement_content": "review_content_engagement",
    "aging_or_declining_visible": "review_for_refresh",
    "general_monitoring": "monitor"
}

queue["archetype_action"] = queue[
    "content_archetype"
].map(archetype_action_map)

# 4. Display top 10 ranked recommendations
display_columns = [
    col for col in [
        "final_rank",
        "content_id",
        "final_refresh_score",
        "best_model_probability",
        "confidence",
        "suggested_action",
        reason_col,
        "content_archetype",
        "archetype_action"
    ]
    if col in queue.columns
]

print("\nTOP 10 RANKED CONTENT ACTIONS")

display(queue[display_columns].head(10))

# 5. Summarize reason codes
print("\nREASON CODE SUMMARY")

reason_summary = (
    queue[reason_col]
    .fillna("no_reason_recorded")
    .astype(str)
    .str.split("|")
    .explode()
    .str.strip()
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="page_count")
)

display(reason_summary.head(10))

# 6. Review content age and decay signals
age_col = next(
    (
        col for col in [
            "content_age_days",
            "days_since_last_update"
        ]
        if col in queue.columns
    ),
    None
)

if age_col:

    print("\nCONTENT AGE / FRESHNESS SUMMARY")

    print("Age field used:", age_col)

    print(queue[age_col].describe().round(2))

    if age_col == "content_age_days":

        queue["age_band"] = pd.cut(
            queue[age_col],
            bins=[-1, 180, 365, 730, float("inf")],
            labels=[
                "0-180 days",
                "181-365 days",
                "366-730 days",
                "Above 730 days"
            ]
        )

        age_summary = queue.groupby(
            "age_band",
            observed=False
        ).size().reset_index(name="page_count")

        display(age_summary)

else:
    print(
        "\nNo content age field found. "
        "Age-based decay analysis needs an age or update-date feature."
    )

# 7. Final checks
print("\nCONTENT ARCHETYPE COUNTS")

display(
    queue["content_archetype"]
    .value_counts()
    .to_frame("page_count")
)

assert "content_archetype" in queue.columns
assert "archetype_action" in queue.columns

print("\nSection 1 remaining analysis completed successfully.")


Reason-code columns found: ['final_reason_codes']

TOP 10 RANKED CONTENT ACTIONS


,final_rank,content_id,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,content_archetype,archetype_action
0,1,content_1f080331fa2b,81.734212,0.783472,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,high_visibility_low_ctr,review_title_and_description
1,2,content_d6570c51c9bd,81.603243,0.849842,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
2,3,content_6aa43079fb0c,81.544618,0.789490,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
3,4,content_72e800a9c214,81.169731,0.776297,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
4,5,content_e04eb9549989,80.957565,0.816010,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
5,6,content_b69288c5e701,80.798090,0.796332,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
6,7,content_9b6df29f7889,80.650656,0.846499,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
7,8,content_ba6f9dfcbca1,80.432641,0.827496,medium,refresh,declining_with_demand|model_decline_risk|visib...,aging_or_declining_visible,review_for_refresh
8,9,content_4d76cdb3387b,80.428403,0.844030,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high_visibility_low_ctr,review_title_and_description
9,10,content_b4f35d640b1c,80.428243,0.845325,medium,refresh,declining_with_demand|model_decline_risk|visib...,aging_or_declining_visible,review_for_refresh



REASON CODE SUMMARY


,reason_code,page_count
0,declining_with_demand,13152
1,visible_model_opportunity,10650
2,low_ctr_visible_page,9759
3,ctr_review_candidate,9759
4,general_refresh_review,9117
5,model_decline_risk,8427
6,page_one_decay_risk,7076
7,low_engagement_visible_page,6508
8,engagement_review_candidate,6508
9,thin_visible_page,82



CONTENT AGE / FRESHNESS SUMMARY
Age field used: content_age_days
count    30000.00
mean       256.17
std        132.71
min         90.00
25%        132.00
50%        236.00
75%        333.00
max        564.00
Name: content_age_days, dtype: float64


,age_band,page_count
0,0-180 days,12272
1,181-365 days,11368
2,366-730 days,6360
3,Above 730 days,0



CONTENT ARCHETYPE COUNTS


,page_count
content_archetype,
high_visibility_low_ctr,9741
general_monitoring,9117
aging_or_declining_visible,7645
low_engagement_content,3415
thin_visible_content,82



Section 1 remaining analysis completed successfully.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


## 2. Intended use and limits

### Intended use

The content action playbook is designed for content strategists, SEO analysts, and editorial teams.

It helps them:

- Prioritize pages for manual review.
- Identify possible content freshness and visibility issues.
- Find pages with possible CTR or engagement opportunities.
- Understand why a page received a particular recommendation.
- Organize a large content review backlog.

### Limits

The model was developed using an anonymized starter dataset containing 30,000 content records.

The declining label is based on the observed trend_direction field. It represents the historical label used in the experiment and does not guarantee future decline.

The Week-6 validation comparison showed:

- Random row split Precision@50: 0.90.
- Client-grouped split Precision@50: 0.54.

This difference shows why results from unseen clients should be considered more carefully than random row results.

The model does not prove that refreshing content will increase traffic, improve rankings, or change Google's algorithm.

The output is decision-support information, not an automatic publishing system. It should not be treated as a production-ready recommendation engine.


In [10]:

import json
import pandas as pd
from pathlib import Path

# Load official model results
metrics_path = repo_path / "outputs" / "model_results.json"

if not metrics_path.exists():
    raise FileNotFoundError("Model results JSON not found.")

with open(metrics_path, "r") as file:
    model_results = json.load(file)

best_model_name = model_results["best_model"]["name"]

best_metrics = model_results["models"][best_model_name]

print("Official pipeline validation information")
print("Best model:", best_model_name)
print("Split strategy:", model_results["split_strategy"])
print("Target:", model_results["target"])

print("\nOfficial model metrics:")

official_metrics = pd.DataFrame([{
    "Model": best_model_name,
    "Precision@50": best_metrics["precision_at_50"],
    "ROC-AUC": best_metrics["roc_auc"],
    "Average Precision": best_metrics["average_precision"]
}])

display(official_metrics.round(4))

# Week-6 additional random versus grouped comparison
week6_results = pd.DataFrame([
    {
        "Validation": "Random row split",
        "Precision@50": 0.90
    },
    {
        "Validation": "Client-grouped split",
        "Precision@50": 0.54
    }
])

print("\nWeek-6 additional validation comparison:")
display(week6_results)

print(
    "\nInterpretation: The grouped validation result is a more cautious "
    "reference for generalization to unseen clients."
)


Official pipeline validation information
Best model: random_forest
Split strategy: client_holdout
Target: is_declining_label

Official model metrics:


,Model,Precision@50,ROC-AUC,Average Precision
0,random_forest,0.74,0.75,0.6182



Week-6 additional validation comparison:


,Validation,Precision@50
0,Random row split,0.90
1,Client-grouped split,0.54



Interpretation: The grouped validation result is a more cautious reference for generalization to unseen clients.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


## 3. Human review + the no-go list

Every recommendation must be reviewed by a human before action is taken.

### Human review rules

1. Check the page's current content before approving a refresh.
2. Verify whether the page has sufficient search impressions and sessions.
3. Check whether the reason codes match the actual page condition.
4. Review search intent and content relevance.
5. Check whether the content is already updated or has important business information.
6. Confirm the proposed action with the content owner.
7. Record the final human decision.

High-confidence pages should receive early attention, but high confidence does not mean that the recommendation is automatically correct.

### What should NOT be automated

The system must never automatically:

- Delete content pages.
- Redirect URLs.
- Publish or rewrite content.
- Change legal, medical, or policy-related content.
- Modify website navigation or URL structures.
- Change canonical or indexing settings.
- Make business-critical editorial decisions.

The model only prioritizes content for review. Final decisions remain with the responsible human team.


In [11]:

import pandas as pd

# Load the ranked queue
review_queue = pd.read_csv(
    repo_path / "outputs" / "refresh_queue.csv"
)

# Every recommendation requires human review
review_queue["human_review_required"] = True

# Assign review priority
high_score_threshold = review_queue[
    "final_refresh_score"
].quantile(0.80)

review_queue["human_review_priority"] = review_queue.apply(
    lambda row: (
        "priority_review"
        if (
            row["confidence"] == "high"
            or row["final_refresh_score"] >= high_score_threshold
        )
        else "standard_review"
    ),
    axis=1
)

print("Human review summary:")

display(
    review_queue["human_review_priority"]
    .value_counts()
    .to_frame("page_count")
)

print(
    "\nAll recommendations requiring human review:",
    review_queue["human_review_required"].sum()
)

# No-go actions
no_go_actions = [
    "Automatic content deletion",
    "Automatic URL redirects",
    "Automatic content publishing",
    "Automatic legal content modification",
    "Automatic navigation changes",
    "Automatic canonical or indexing changes"
]

print("\nActions that must never be automated:")

for action in no_go_actions:
    print("-", action)

print("\nSection 3 human-review checks completed.")



Human review summary:


,page_count
human_review_priority,
standard_review,24000
priority_review,6000



All recommendations requiring human review: 30000

Actions that must never be automated:
- Automatic content deletion
- Automatic URL redirects
- Automatic content publishing
- Automatic legal content modification
- Automatic navigation changes
- Automatic canonical or indexing changes

Section 3 human-review checks completed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


## 4. Monitoring / retrain triggers

The action playbook should be reviewed regularly because content performance and search behaviour can change over time.

### Monitoring plan

- Review the ranked queue every month.
- Check whether the top-ranked pages still have valid reasons for review.
- Monitor changes in impressions, CTR, sessions, and engagement.
- Check whether the distribution of model scores changes significantly.
- Compare model performance against newly available labelled data.

### Retraining triggers

Retraining should be considered when:

1. New reliable labelled data becomes available.
2. The feature definitions or target definition changes.
3. The distribution of important input features changes significantly.
4. Precision@50 on a fresh client-grouped validation set falls below the agreed monitoring threshold.
5. Repeated human reviews show that recommendations are no longer useful.

A proposed monitoring threshold is a 20% relative decline from the Week-6 grouped Precision@50 of 0.54.

This gives a trigger value of approximately 0.432.

These thresholds are proposed monitoring rules, not measured production limits.

Retraining should only happen after checking data quality, leakage, and validation performance again.


In [12]:

import pandas as pd

# Proposed monitoring settings
week6_grouped_precision = 0.54

# 20% relative decline threshold
precision_warning_threshold = week6_grouped_precision * 0.80

monitoring_plan = pd.DataFrame([
    {
        "Monitoring item": "Ranked queue review",
        "Frequency": "Monthly",
        "Trigger": "Review top-ranked recommendations"
    },
    {
        "Monitoring item": "Input feature distribution",
        "Frequency": "Monthly",
        "Trigger": "Investigate significant distribution changes"
    },
    {
        "Monitoring item": "Model Precision@50",
        "Frequency": "When new labels become available",
        "Trigger": f"Below {precision_warning_threshold:.3f}"
    },
    {
        "Monitoring item": "Target definition",
        "Frequency": "Whenever definitions change",
        "Trigger": "Rebuild and validate the model"
    },
    {
        "Monitoring item": "Human review feedback",
        "Frequency": "Monthly",
        "Trigger": "Repeated unsuitable recommendations"
    }
])

print("PROPOSED MONITORING PLAN")
display(monitoring_plan)

print(
    "\nWeek-6 grouped Precision@50:",
    week6_grouped_precision
)

print(
    "Proposed warning threshold:",
    round(precision_warning_threshold, 3)
)

print(
    "\nThese are proposed monitoring triggers, "
    "not measured production thresholds."
)



PROPOSED MONITORING PLAN


,Monitoring item,Frequency,Trigger
0,Ranked queue review,Monthly,Review top-ranked recommendations
1,Input feature distribution,Monthly,Investigate significant distribution changes
2,Model Precision@50,When new labels become available,Below 0.432
3,Target definition,Whenever definitions change,Rebuild and validate the model
4,Human review feedback,Monthly,Repeated unsuitable recommendations



Week-6 grouped Precision@50: 0.54
Proposed warning threshold: 0.432

These are proposed monitoring triggers, not measured production thresholds.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


## 5. Exports for the paper

The final ranked content action queue will be exported to `work/outputs/`.

The exported queue includes:

- Content rank.
- Final refresh score.
- Model probability.
- Confidence level.
- Suggested action.
- Reason codes.
- Content archetype.
- Archetype-to-action mapping.
- Relevant performance indicators.

Pseudonymized content IDs are retained for reference. Client identifiers and private identifying information are excluded.

The generated figures will be copied into `work/figures/` for reuse in the research paper.

The model metrics JSON will also be copied into the working outputs folder as a reproducibility record.

The ranked CSV is a generated artifact and should not be committed to GitHub. The notebook, reusable figures, and required metrics JSON should be committed.

All exported recommendations remain intended for human review.


In [13]:

import json
import shutil
import pandas as pd
from pathlib import Path

# Create export directories
work_output_dir = repo_path / "work" / "outputs"
work_figure_dir = repo_path / "work" / "figures"

work_output_dir.mkdir(parents=True, exist_ok=True)
work_figure_dir.mkdir(parents=True, exist_ok=True)

# Load the official final queue
source_queue = pd.read_csv(
    repo_path / "outputs" / "refresh_queue.csv"
)

# Add archetypes and actions
def get_archetype(reason_text):

    reasons = set(str(reason_text).split("|"))

    if "thin_visible_page" in reasons:
        return "thin_visible_content"

    if {
        "ctr_review_candidate",
        "low_ctr_visible_page"
    }.intersection(reasons):
        return "high_visibility_low_ctr"

    if {
        "engagement_review_candidate",
        "low_engagement_visible_page"
    }.intersection(reasons):
        return "low_engagement_content"

    if {
        "stale_visible_page",
        "declining_with_demand",
        "model_decline_risk"
    }.intersection(reasons):
        return "aging_or_declining_visible"

    return "general_monitoring"


action_mapping = {
    "thin_visible_content": "expand_and_refresh",
    "high_visibility_low_ctr": "refresh_and_review_ctr",
    "low_engagement_content": "refresh_and_review_engagement",
    "aging_or_declining_visible": "refresh",
    "general_monitoring": "monitor"
}

source_queue["content_archetype"] = (
    source_queue["final_reason_codes"].apply(get_archetype)
)

source_queue["archetype_action"] = (
    source_queue["content_archetype"].map(action_mapping)
)

source_queue["human_review_required"] = True

# Select public-safe columns
safe_columns = [
    "final_rank",
    "content_id",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "content_archetype",
    "archetype_action",
    "human_review_required",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "content_type",
    "main_intent"
]

export_columns = [
    col for col in safe_columns
    if col in source_queue.columns
]

ranked_export = source_queue[export_columns].copy()

# Export ranked queue
queue_export_path = (
    work_output_dir / "refresh_action_queue.csv"
)

ranked_export.to_csv(queue_export_path, index=False)

# Copy official model metrics JSON
metrics_source = repo_path / "outputs" / "model_results.json"

metrics_export_path = (
    work_output_dir / "ml10_model_metrics.json"
)

if metrics_source.exists():
    shutil.copy2(metrics_source, metrics_export_path)
else:
    raise FileNotFoundError("Model metrics JSON not found.")

# Copy reusable charts
charts_source = repo_path / "outputs" / "charts"

copied_figures = []

if charts_source.exists():

    for figure in charts_source.glob("*.svg"):

        destination = work_figure_dir / figure.name

        shutil.copy2(figure, destination)

        copied_figures.append(destination)

# Export summary information
summary = {
    "total_ranked_pages": int(len(ranked_export)),
    "best_model": str(ranked_export["best_model_name"].iloc[0]),
    "action_counts": {
        str(k): int(v)
        for k, v in ranked_export["suggested_action"].value_counts().items()
    },
    "queue_file": "work/outputs/refresh_action_queue.csv",
    "metrics_file": "work/outputs/ml10_model_metrics.json",
    "figures_exported": [p.name for p in copied_figures],
    "human_review_required": True
}

summary_path = work_output_dir / "ml10_playbook_summary.json"

with open(summary_path, "w") as file:
    json.dump(summary, file, indent=4)

# Verify exports
print("EXPORT VERIFICATION")

for path in [
    queue_export_path,
    metrics_export_path,
    summary_path
]:
    print(
        path.relative_to(repo_path),
        "| Exists:", path.exists(),
        "| Size:", path.stat().st_size if path.exists() else 0
    )

print("\nFigures copied:")

for figure in copied_figures:
    print(figure.relative_to(repo_path))

# Check that client_id is excluded
assert "client_id" not in ranked_export.columns

# Check that private URL and target columns are excluded
assert not any(
    "url" in col.lower()
    for col in ranked_export.columns
)

assert "is_declining_label" not in ranked_export.columns

print("\nAll export checks passed.")
print("Section 5 completed successfully.")



EXPORT VERIFICATION
work/outputs/refresh_action_queue.csv | Exists: True | Size: 7763959
work/outputs/ml10_model_metrics.json | Exists: True | Size: 5029
work/outputs/ml10_playbook_summary.json | Exists: True | Size: 617

Figures copied:
work/figures/confidence_mix.svg
work/figures/top_feature_importance.svg
work/figures/action_mix.svg
work/figures/trend_distribution.svg
work/figures/top_reason_codes.svg

All export checks passed.
Section 5 completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.